**Project : Spacecraft Charging model that includes Distribution Function of Plasma Particles (Electrons and Ions) from magnetic reconnection phenomena in space - Computing temporal variation of spacecraft Charging Potential**

The cells are arranged in following orders : 
1. Define Physical Constants and Plasma Parameters 
2. Define functions that are essential for final goal
3. Required Prints and plots 

In [1]:
import numpy as np
import matplotlib.pyplot as plt

# Physical Constants 
# ==========================================================
# 1. Physical constants 

kB = 1.38e-23  # Boltzmann constant in J/K
me = 9.11e-31  # Electron mass in kg
mi= 1.67e-27  # Ion mass (proton) in kg
qe = -1.60e-19  # Electron charge in Coulombs
qi = 1.60e-19  # Ion charge in Coulombs

# ==========================================================
#Plasma parameters
# ==========================================================

Te = 1.0e6                       # in Kelvin
Ti= Te                           # in Kelvin
beta = 1.0 / (kB * Te)
vth = np.sqrt(1.0 / (me * beta)) # Thermal velocity of electrons
vTi = np.sqrt(1.0 / (mi * beta)) # Thermal velocity of ions
n0 = 1.0 * 1e6                   # Plasma density in m^-3
b = 100                          # Magnetic field strength in nT
u0 = 5 * vth                     # Bulk flow velocity in m/s

# ==========================================================
#Magnetic field parameters
# ==========================================================

B0 = 5e-6  # Magnetic field strength in Tesla
L  = 1.0   # Characteristic length scale in meters
z  = 2.0 * L  #


# ==========================================================
# Spacecraft parameters
# ==========================================================

A = 29.68             # m^2 (For time being we are considering a surface area of 29.68 m^2, which is the surface area from a paper by Electrostatic Analysis of an Artificial Orbiting Satellite for Absolute Charging Rizwan H. Alad and Soumyabrata Chakrabarty, Senior Member, IEEE, we can change this later based on the actual spacecraft we want to model)
C = 328.86e-12     # F      (same paper as above, we can change this later based on the actual spacecraft we want to model)




In [ ]:
#Magnetic vector Potential Components


def Ax(z):
    '''
    Calculate the x-component of the magnetic vector potential.
    From the given magnetic vector potential, we can derive the magnetic field components and vice versa. These components are essential in understanding and deriving the Distribution functions of electrons and ions in the plasma, which are crucial for computing the currents and ultimately the spacecraft potential.
    
    :param z: The Harris sheet is a model of a current sheet in plasma physics, where the magnetic field changes direction across the sheet. The z-coordinate is typically used to describe the position across the sheet, with z=0 being the center of the sheet. The magnetic vector potential components Ax and Ay are functions of z, and they help describe the magnetic field configuration in this model.
    return B0 * L * np.arctan(np.sinh(z / L))

    '''
    return B0 * L * np.arctan(np.sinh(z / L))

def Ay(z):
    '''
    Calculate the y-component of the magnetic vector potential.
    The y-component of the magnetic vector potential, Ay, is also a function of the z-coordinate in the Harris sheet model. It is derived from the magnetic field configuration and is essential for understanding the behavior of charged particles in the plasma, which in turn affects the currents and spacecraft charging.
    return B0 * L * np.log(np.cosh(z / L))

    '''
    return -B0 * L * np.log(np.cosh(z / L))




The expression for Distribution function contains the Hamiltonian of the two system (Electrons and ions) that is given by 
$$
H_s = \frac{m_s v^2}{2} + q_s \Phi
$$

The s here defines the type of species (in our case either electron or ions), canonical momentum given by 
$$
p_{x,s} = m_s v_x + q_s A_x, \quad
p_{y,s} = m_s v_y + q_s A_y
$$

In the next cell we will define functions that are relavant in finally having the DF. The DF for electrons and ions are given respectively by
$$
F_e(H_e, p_{x,e}, p_{y,e}) =
\frac{n_0}{(1+b)(\sqrt{2\pi} v_{th,e})^3}
e^{-\beta_e H_e}
\left[
b - \frac{1}{2}\beta_e m_e u_0^2 \cos(\beta_e u_0 p_{x,e})
+ e^{\beta_e m_e u_0^2 / 2} e^{\beta_e u_0 p_{y,e}}
\right]
\tag{elecDF}
$$

$$
F_i(H_i, p_{x,i}, p_{y,i}) =
\frac{n_{0,i}}{(\sqrt{2\pi} v_{th,i})^3}
e^{-\beta_i H_i}
\tag{ionDF}
$$




In [ ]:
# ==========================================================

# To get the DF, we need to define the Hamilonian of the system for which velocity need to defined
# Velocity grid 

v = np.linspace(-5*vth, 5*vth, 1000) # this range is taken in such a way that it covers the significant part of the Maxwellian distribution, actual range can be adjusted based on the specific plasma conditions and the expected particle velocities.

#==========================================================

#The Hamiltonian of the system consists the term $phi$ that is electrostatic potential due to plasma environment
# The paper we are following not directly considered phi to be 0 but expanded it in Taylor series and considered only the first term.
# First-order electrostatic potential (Phi1) from analytical expansion in the paper.
# The equilibrium is not strictly neutral, so Phi is introduced as a small correction.
# Instead of solving Poisson numerically, we use the derived relation:
# e*beta_e*Phi1 ∝ Ax → Phi1 is directly coupled to the magnetic vector potential.
# Zeroth-order potential Phi0 = 0, so total Phi ≈ Phi1.

def Phi1(z):

    '''
    First order correction to Phi(electrostatic potential) due to the plasma environment, derived from the analytical expansion in the paper.
    return -2 * Ax(z) / (qe * beta * (b + 0.5) * (1 + beta / beta))
    '''
    coeff = -2 / (qe * beta * (b + 0.5) * (1 + beta / beta))
    return coeff * Ax(z) / (B0 * L)

# ==================================================
# Electron DF in x-direction
# Set vy = vz = 0
# ==================================================
vx = v
vy = 0.0
vz = 0.0

# Hamiltonian in x-direction for electrons
He_x = 0.5 * me * (vx**2 + vy**2 + vz**2) + qe * Phi1(z)

# Canonical momentum px in x-direction for electrons
px = me * vx + qe * Ax(z)

# Electron distribution function in x-direction

def Fe_x(He_x, px, n0, b, vth, beta, me, u0):
    '''
    This is the electron distribution function in the x-direction, derived from the Hamiltonian and canonical momentum. It incorporates the effects of the electrostatic potential and magnetic vector potential, which are crucial for understanding the behavior of electrons in the plasma and their contribution to the currents that affect spacecraft charging.
    Parameters: Hamiltonian He_x, Canonical momentum px, plasma density n0, magnetic field parameter b, thermal velocity vth, inverse temperature beta, electron mass me, and bulk flow velocity u0.
    '''
    return (
        n0 / ((1 + b) * (np.sqrt(2*np.pi) * vth)**3)
        * np.exp(-beta * He_x)
        * (
            b
            - 0.5 * np.exp(beta * me * u0**2 / 2.0)
              * np.cos(beta * u0 * px)
            + np.exp(beta * me * u0**2 / 2.0)
        )
    )

# ==================================================
# ==================================================
# Electron DF in y-direction
# Set vx = vz = 0
# ==================================================
vx = 0.0
vy = v
vz = 0.0

# Hamiltonian in y-direction for electrons
He_y = 0.5 * me * (vx**2 + vy**2 + vz**2) + qe * Phi1(z)

# Canonical momentum py in y-direction for electrons
py = me * vy + qe * Ay(z)

# Electron DF depending only on vy

def Fe_y(py, He_y,u0, n0, b, vth, beta, me):
    '''
    The electron distribution function in the y-direction, which is derived similarly to the x-direction but incorporates the y-component of the canonical momentum and Hamiltonian. This function is essential for understanding the behavior of electrons in the y-direction and their contribution to the currents that affect spacecraft charging.
    
    :param py: canonical momentum in the y-direction for electrons
    :param He_y: Hamiltonian in the y-direction for electrons
    :param u0: bulk flow velocity in m/s
    :param n0: initial plasma density in m^-3
    :param b: magnetic field strength parameter
    :param vth: thermal velocity of electrons
    :param beta: inverse temperature*boltzmann constant
    :param me: Mass of electron in kg
    :return: The electron distribution function in the y-direction. 
    '''
    return (
        n0 / ((1 + b) * (np.sqrt(2*np.pi) * vth)**3)
        * np.exp(-beta * He_y)
        * (
            b
        - 0.5 * np.exp(beta * me * u0**2 / 2.0)
        + np.exp(beta * me * u0**2 / 2.0)
          * np.exp(beta * u0 * py)
    )
)